In [8]:
from pathlib import Path
from PIL import Image

src_dir_rgb = Path("/home/khang/Documents/VR/bkhn_ver3.0_test/round2_bkhn/videos/chunk-000/observation.images.rgb.125cm_30deg")
dst_dir_rgb = Path("/home/khang/Documents/VR/InternNav/assets/bkhn_data_round2_e15_rgb")
dst_dir_rgb.mkdir(exist_ok=True)

src_dir_depth = Path("/home/khang/Documents/VR/bkhn_ver3.0_test/round2_bkhn/videos/chunk-000/observation.images.depth.125cm_30deg")
dst_dir_depth = Path("/home/khang/Documents/VR/InternNav/assets/bkhn_data_round2_e15_depth")
dst_dir_depth.mkdir(exist_ok=True)

for frame_num, rgb_file in enumerate(sorted(src_dir_rgb.glob("episode_000015_*.jpg")), start=1):
    frame_id = str(int(rgb_file.stem.split("_")[-1])).zfill(4)
    depth_file = src_dir_depth / f"{rgb_file.stem}.png"
    if not depth_file.exists():
        raise FileNotFoundError(f"Missing depth file for {rgb_file.name}: {depth_file}")

    if frame_num % 5 == 0 and frame_num > 0:
        out_name = f"debug_raw_{frame_id}_look_down.jpg"
    else:
        out_name = f"debug_raw_{frame_id}.jpg"

    Image.open(rgb_file).convert("RGB").save(
        dst_dir_rgb / out_name,
        quality=95,
    )
    Image.open(depth_file).convert("RGB").save(
        dst_dir_depth / out_name,
        quality=95,
    )

# print(f"Saved {frame_num} RGB images to {dst_dir_rgb}")
# print(f"Saved {frame_num} depth images to {dst_dir_depth}")



In [2]:
import pandas as pd

path = "/home/khang/Documents/VR/InternNav/data/InternData-N1/vln_ce/traj_data/r2r/1LXtFkjw3qL/data/chunk-000/episode_000000.parquet"
df = pd.read_parquet(path)

setting = "125cm_30deg"
print(df[["action", f"pose.{setting}", f"relative_goal_frame_id.{setting}"]].head(10))
print(f"Episode length: {len(df)} frames")

   action                                   pose.125cm_30deg  \
0      -1  [[0.0, -0.5, 0.866, 0.0], [-1.0, 0.0, 0.0, 0.0...   
1       1  [[0.0, -0.5, 0.866, 0.25], [-1.0, 0.0, 0.0, 0....   
2       1  [[0.0, -0.5, 0.866, 0.5], [-1.0, 0.0, 0.0, 0.0...   
3       1  [[0.0, -0.5, 0.866, 0.75], [-1.0, 0.0, 0.0, 0....   
4       1  [[0.0, -0.5, 0.866, 1.0], [-1.0, 0.0, 0.0, 0.0...   
5       1  [[0.0, -0.5, 0.866, 1.25], [-1.0, 0.0, 0.0, 0....   
6       2  [[0.2588, -0.483, 0.8365, 1.25], [-0.9659, -0....   
7       2  [[0.5, -0.433, 0.75, 1.25], [-0.866, -0.25, 0....   
8       1  [[0.5, -0.433, 0.75, 1.4665], [-0.866, -0.25, ...   
9       2  [[0.7071, -0.3536, 0.6124, 1.4665], [-0.7071, ...   

   relative_goal_frame_id.125cm_30deg  
0                                  14  
1                                  13  
2                                  13  
3                                  11  
4                                  -1  
5                                  -1  
6              

In [ ]:
# pose: 4x4 camera extrinsic matrix (float32)

In [ ]:
# CV
# +X = right 
# +Y = down 
# +Z = forward 

# robotics
# +X = forward
# +Y = left
# +Z = up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "/home/khang/Documents/VR/InternNav")

from internnav.dataset.internvla_n1_lerobot_dataset import (
    get_trajectory_relative_to_frame,
    interpolate_and_resample_trajectory,
)

path = "/home/khang/Documents/VR/InternNav/data/InternData-N1/vln_ce/traj_data/r2r/1LXtFkjw3qL/data/chunk-000/episode_000000.parquet"
setting, pitch = "125cm_30deg", 30

df = pd.read_parquet(path)
poses = np.stack([np.asarray(p, float) for p in df[f"pose.{setting}"]])

# 1) Full episode relative path
traj = get_trajectory_relative_to_frame(poses, camera_deg=pitch)
plt.plot(traj[:, 1], traj[:, 0]); plt.gca().invert_xaxis(); plt.axis("equal")
plt.xlabel("Y"); plt.ylabel("X"); plt.title("Full episode (relative to t=0)")
plt.save
plt.show()

# 2) S1-style 33-point future path from t=0
goal_len = int(df[f"relative_goal_frame_id.{setting}"].iloc[0])
slice_poses = poses[: goal_len + 1]
discrete = get_trajectory_relative_to_frame(slice_poses, camera_deg=pitch)
resampled_xy, _ = interpolate_and_resample_trajectory(discrete, predict_step_num=32)
plt.plot(resampled_xy[:, 1], resampled_xy[:, 0], "r--")
plt.show()

PROJECT_ROOT_PATH:/home/khang/Documents/VR/InternNav


/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'decord'